# Amosclaud Autonomous Engineering Lab

This notebook demonstrates planning, restricted edits, verification, shared database records, and the signed Byte bus. It is a development laboratory only: it uses temporary storage and must not be connected to production secrets or live repositories.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json

from agents import RealCodexAgent
from Amosclaud.platform_bus import PlatformByteBus
from database.models import AutonomousJob, AutonomousJobStatus, CIPipeline, CIStatus, Repository, UserProfile
from database.session import create_database, session_scope


## 1. Inspect the enforced agent policy

In [ ]:
manifest = json.loads(Path('../agents/manifest.json').read_text(encoding='utf-8'))
manifest


## 2. Create isolated platform records
The lab creates a user, repository, CI pipeline, and Autonomous job in the configured development database. Use a temporary `AMOSCLAUD_PLATFORM_DATABASE_URL` before running this notebook.

In [ ]:
create_database()
with session_scope() as session:
    user = UserProfile(username='lab-user', email='lab-user@example.test')
    repository = Repository(name='lab-repository', owner=user, default_branch='main')
    session.add(repository)
    session.flush()
    pipeline = CIPipeline(repository=repository, commit_sha='0' * 40, status=CIStatus.PENDING)
    session.add(pipeline)
    session.flush()
    job = AutonomousJob(task_id='lab-repair-1', agent_type='verified-software-worker', repository=repository, ci_pipeline=pipeline, objective='Create and verify a small Python module', status=AutonomousJobStatus.QUEUED)
    session.add(job)
    repository_id, pipeline_id = repository.id, pipeline.id
repository_id, pipeline_id


## 3. Exercise the signed internal Byte bus

In [ ]:
bus = PlatformByteBus(b'development-lab-secret-not-for-production')
bus.execute(bus.frame('platform.repository.summary', {'repository_id': repository_id})).json()


## 4. Run the restricted worker in a temporary workspace

In [ ]:
class DemonstrationModel:
    def __init__(self):
        self.responses = [
            '```write:src/example.py\nVALUE = 1\n```',
            '```execute\npython -m compileall -q .\n```',
        ]
    def generate(self, history):
        return self.responses.pop(0)

with TemporaryDirectory() as folder:
    workspace = Path(folder)
    worker = RealCodexAgent(str(workspace), DemonstrationModel(), platform_bus=bus, task_id='lab-repair-1')
    result = worker.run_task('Create and verify a small Python module')
    print(result)
    print((workspace / 'src' / 'example.py').read_text())


## 5. Inspect persisted verification evidence

In [ ]:
with session_scope() as session:
    job = session.query(AutonomousJob).filter_by(task_id='lab-repair-1').one()
    pipeline = session.get(CIPipeline, pipeline_id)
    evidence = {
        'job_status': job.status.value,
        'ci_status': pipeline.status.value,
        'verification_id': pipeline.verification_id,
        'summary': job.result_summary,
    }
evidence


## Safety rules

- Never place production API keys, cookies, database URLs, or repository secrets in this notebook.
- Never point the worker at a live user repository.
- Never treat notebook output as merge approval. CI remains authoritative.
- Use normal Python modules and tests for production behavior; this notebook is for explanation and experiments.